Šis užrašų knygelės failas buvo automatiškai sugeneruotas GitHub Copilot Chat ir skirtas tik pradiniam nustatymui


# Įvadas į užklausų inžineriją
Užklausų inžinerija yra procesas, kurio metu projektuojamos ir optimizuojamos užklausos natūralios kalbos apdorojimo užduotims. Tai apima tinkamų užklausų parinkimą, jų parametrų reguliavimą ir našumo vertinimą. Užklausų inžinerija yra labai svarbi norint pasiekti aukštą tikslumą ir efektyvumą NLP modeliuose. Šiame skyriuje nagrinėsime užklausų inžinerijos pagrindus, naudojant OpenAI modelius tyrimams.


### Užduotis 1: Tokenizacija
Išnagrinėkite tokenizaciją naudodami tiktoken, atviro kodo greitą tokenizatorių iš OpenAI
Daugiau pavyzdžių žr. [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst).


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### Užduotis 2: Patvirtinkite OpenAI API rakto nustatymą

Paleiskite žemiau esantį kodą, kad patikrintumėte, ar jūsų OpenAI galinis taškas yra tinkamai nustatytas. Kodas tiesiog bando paprastą pagrindinį užklausimą ir patikrina užbaigimą. Įvestis `oh say can you see` turėtų būti užbaigta maždaug taip: `by the dawn's early light..`


In [ ]:
# Uses the OpenAI client against the Azure OpenAI (Microsoft Foundry) v1 endpoint
# with the Responses API. See https://aka.ms/openai/start

import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']

def get_completion(prompt):
    response = client.responses.create(
        model=deployment,
        input=prompt,
        max_output_tokens=1024,
        store=False,
    )
    return response.output_text

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt)
print(response)


### Užduotis 3: Išgalvotos istorijos
Ištirkite, kas nutinka, kai prašote LLM pateikti užbaigimus užklausai apie temą, kuri gali neegzistuoti, arba apie temas, apie kurias jis gali nežinoti dėl to, kad jos nebuvo jo išankstinio mokymo duomenų rinkinyje (naujesnės). Pažiūrėkite, kaip atsakymas keičiasi, jei bandote kitą užklausą arba kitą modelį.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt)
print(response)

### Pratimai 4: Instrukcijomis pagrįsta 
Naudokite kintamąjį "text", kad nustatytumėte pagrindinį turinį 
ir kintamąjį "prompt", kad pateiktumėte su tuo pagrindiniu turiniu susijusią instrukciją.

Čia modelis prašomas santrumpinti tekstą antros klasės mokiniui


In [ ]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt)
print(response)

### Pratimai 5: Sudėtingas užklausimas 
Išbandykite užklausą, kurioje yra sistemos, vartotojo ir asistento pranešimai 
Sistema nustato asistento kontekstą
Vartotojo ir asistento pranešimai suteikia daugkartinio pokalbio kontekstą

Atkreipkite dėmesį, kaip asistento asmenybė yra nustatyta kaip „sarkastiška“ sistemos kontekste. 
Išbandykite naudoti kitokią asmenybės kontekstą. Arba išbandykite kitą įėjimo/išėjimo pranešimų seką


In [ ]:
response = client.responses.create(
    model=deployment,
    input=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ],
    store=False,
)
print(response.output_text)


### Pratimai: Tyrinėkite savo intuiciją
Aukščiau pateikti pavyzdžiai suteikia jums raštus, kuriuos galite naudoti kurdami naujus uždavinius (paprastus, sudėtingus, su instrukcijomis ir kt.) - pabandykite sukurti kitų pratimų, kad ištirtumėte kai kurias kitas idėjas, apie kurias kalbėjome, tokias kaip pavyzdžiai, užuominos ir daugiau.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
